# Phân tích Dữ liệu Khám phá (EDA) & Tiền xử lý
**Dự án:** Hệ thống gợi ý phim cho người dùng Việt Nam (Movie-Recsys-VN)

**Mục tiêu:**
1. Khám phá đặc tính của bộ dữ liệu MovieLens 25M.
2. Xử lý dữ liệu thô thành Phản hồi ngầm định (Implicit Feedback).
3. Làm giàu dữ liệu bằng Metadata từ TMDB và trích xuất đặc trưng tiếng Việt.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Cấu hình đồ thị
plt.style.use('seaborn-v0_8')
%matplotlib inline

DATA_RAW_PATH = "../../data/raw/"
DATA_PROCESSED_PATH = "../../data/processed/"

os.makedirs(DATA_PROCESSED_PATH, exist_ok=True)

In [ ]:
# MovieLens 25M
ratings = pd.read_csv(DATA_RAW_PATH + 'ratings.csv')
movies = pd.read_csv(DATA_RAW_PATH + 'movies.csv')
links = pd.read_csv(DATA_RAW_PATH + 'links.csv')

print(f"Số lượng Ratings: {len(ratings):,}")
print(f"Số lượng Movies: {len(movies):,}")

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x='rating', data=ratings)
plt.title('Phân phối điểm đánh giá (Ratings)')
plt.show()

n_users = ratings['userId'].nunique()
n_items = ratings['movieId'].nunique()
sparsity = 100 * (1 - len(ratings) / (n_users * n_items))
print(f"Độ thưa của ma trận: {sparsity:.4f}%")

movie_counts = ratings.groupby('movieId').size().sort_values(ascending=False)
plt.figure(figsize=(10, 4))
plt.plot(movie_counts.values)
plt.title('Phân phối Long-tail của các bộ phim')
plt.xlabel('Thứ tự phim')
plt.ylabel('Số lượng tương tác')
plt.show()

In [ ]:
# Chỉ giữ lại các rating từ 4.0 trở lên là "Thích" 
ratings = ratings[ratings['rating'] >= 4.0]

# Loại bỏ người dùng có ít hơn 20 tương tác để giảm nhiễu
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts >= 20].index
ratings = ratings[ratings['userId'].isin(active_users)]

print(f"Dữ liệu sau khi lọc: {len(ratings):,} tương tác")

In [ ]:
data_merged = pd.merge(ratings, links, on='movieId', how='inner')

movie_metadata = movies[['movieId', 'title', 'genres']]
final_data = pd.merge(data_merged, movie_metadata, on='movieId', how='inner')

final_data.head()

In [ ]:
final_data.to_pickle(DATA_PROCESSED_PATH + 'processed_ratings.pkl')
print("Đã lưu dữ liệu sạch vào thư mục data/processed/")